# Beyond Matplotlib

Options include:

 - [Plot.ly](https://plotly.com/)
 - [Bokeh](https://docs.bokeh.org/en/latest/index.html)
 - [TikZ](https://texample.net/tikz/)

In [7]:
import numpy as np
import pandas 
import os
import urllib.request
if not os.path.isfile('./covid.csv'):
    urllib.request.urlretrieve("https://api.covidtracking.com/v1/states/daily.csv", "./covid.csv")
covid = pandas.read_csv('./covid.csv')
# by default, date is a number like 20201030.  Use datetime
# to turn that into one integer that represents the number
# of days since the "epoch" (January 1, 1970)
from datetime import date
todate = lambda x: (date(*(int(str(x)[:4]), int(str(x)[4:6]), int(str(x)[6:]))).toordinal())
days = np.array(list(map(todate, covid.date)))
days = days - min(days)
covid['days'] = days 

In [8]:
import numpy as np
from bokeh.io import show
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, RangeTool
from bokeh.plotting import figure

# --- Example data ---
days = np.arange(0, 300)
positive = np.cumsum(np.random.poisson(10, size=300))
source = ColumnDataSource(data=dict(days=days, positive=positive))

# --- Main plot ---
p = figure(
    height=300, width=800,
    tools="xpan", toolbar_location=None,
    x_axis_location="above",
    background_fill_color="white",
    x_range=(0, 100),
)

p.line('days', 'positive', source=source, line_width=2, color="navy")
p.yaxis.axis_label = "# of positive cases in KS"
p.xaxis.axis_label = "Day"

# --- Range tool plot ---
select = figure(
    title="Drag the middle and edges of the selection box to change the range above",
    height=130, width=800,
    y_range=p.y_range,
    x_axis_type="linear", y_axis_type=None,
    tools="", toolbar_location=None,
    background_fill_color="#efefef",
)

range_tool = RangeTool(x_range=p.x_range)
range_tool.overlay.fill_color = "navy"
range_tool.overlay.fill_alpha = 0.2

select.line('days', 'positive', source=source, color="navy")
select.ygrid.grid_line_color = None
select.add_tools(range_tool)

show(column(p, select))


In [10]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import plotly.express as px

fig = px.line(covid, x="days", y="positive", color="state", log_y=True)

# strip down the rest of the plot
fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    margin=dict(t=10,l=10,b=10,r=10)
)

# disable the modebar for such a small plot
fig.show(config=dict(displayModeBar=False))
